# 🕵️ Deepfake Image Detector - Training Notebook

**BUET CSE Level-3 Project**

This notebook trains a high-accuracy deepfake detector using:
- **EfficientNetB0** (Transfer Learning)
- **CIFAKE Dataset** (Real vs AI-Generated Images)
- **Optimized for Kaggle/Colab Free Tier**

### Key Features:
✅ Memory-efficient pipeline (No RAM crashes)  
✅ Data augmentation for better generalization  
✅ Class weight balancing  
✅ Progressive training strategy  
✅ Proper label verification

In [ ]:
# =================================================================================
# 1. IMPORTS & SETUP
# =================================================================================
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras import mixed_precision
import matplotlib.pyplot as plt
import numpy as np
import os
import gc

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

# Clear Memory
tf.keras.backend.clear_session()
gc.collect()

# Enable Mixed Precision for faster training
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print("✅ Mixed Precision Enabled")

In [ ]:
# =================================================================================
# 2. HARDWARE DETECTION (TPU/GPU/CPU)
# =================================================================================
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    print('🔥 TPU Device:', tpu.master())
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print("✅ TPU Strategy Activated")
except:
    strategy = tf.distribute.get_strategy()
    print("✅ Using Default Strategy (GPU/CPU)")

print(f"Number of devices: {strategy.num_replicas_in_sync}")

In [ ]:
# =================================================================================
# 3. CONFIGURATION
# =================================================================================
IMG_SIZE = 224
BATCH_SIZE = 32  # Optimized for free tier
EPOCHS = 15      # Enough for good accuracy
FINE_TUNE_EPOCHS = 10

# Dataset Path Detection (Works on Kaggle & Colab)
DATASET_PATH = '/kaggle/input/cifake-real-and-ai-generated-synthetic-images'

# Fallback for different dataset locations
if not os.path.exists(DATASET_PATH):
    possible_paths = [
        '/content/cifake',
        '/content/drive/MyDrive/cifake',
        './cifake'
    ]
    for path in possible_paths:
        if os.path.exists(path):
            DATASET_PATH = path
            break
    
    # Last resort: search for train folder
    if not os.path.exists(DATASET_PATH):
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'train' in dirs:
                DATASET_PATH = root
                break

print(f"📁 Dataset Path: {DATASET_PATH}")
TRAIN_DIR = os.path.join(DATASET_PATH, 'train')
TEST_DIR = os.path.join(DATASET_PATH, 'test')

# Verify paths
if os.path.exists(TRAIN_DIR):
    print(f"✅ Train directory found: {TRAIN_DIR}")
    print(f"   Classes: {os.listdir(TRAIN_DIR)}")
else:
    print("❌ Train directory not found!")
    
if os.path.exists(TEST_DIR):
    print(f"✅ Test directory found: {TEST_DIR}")
else:
    print("⚠️ Test directory not found!")

In [ ]:
# =================================================================================
# 4. DATA AUGMENTATION (Critical for 99%+ Accuracy)
# =================================================================================
# Data augmentation helps model generalize better and prevents overfitting

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
    tf.keras.layers.RandomBrightness(0.1),
], name='data_augmentation')

# Preprocessing for EfficientNet
preprocess_input = tf.keras.applications.efficientnet.preprocess_input

print("✅ Data Augmentation Pipeline Created")

In [ ]:
# =================================================================================
# 5. DATA LOADING (Memory-Efficient Pipeline)
# =================================================================================
print("📊 Loading Datasets...")

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    label_mode='binary',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=123,
    validation_split=0.2,
    subset='training'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    label_mode='binary',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=123,
    validation_split=0.2,
    subset='validation'
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    label_mode='binary',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False
)

# ⚠️ CRITICAL: Check class names and label mapping
class_names = train_ds.class_names
print(f"\n🏷️ Class Names: {class_names}")
print(f"   Label 0 → {class_names[0]}")
print(f"   Label 1 → {class_names[1]}")

# Calculate class distribution for balancing
total_samples = 0
class_counts = {}

for images, labels in train_ds:
    for label in labels.numpy():
        label_name = class_names[int(label)]
        class_counts[label_name] = class_counts.get(label_name, 0) + 1
        total_samples += 1

print(f"\n📊 Training Set Distribution:")
for class_name, count in class_counts.items():
    print(f"   {class_name}: {count} samples ({count/total_samples*100:.2f}%)")

# Calculate class weights for balanced training
if len(class_counts) == 2:
    class_0_count = class_counts[class_names[0]]
    class_1_count = class_counts[class_names[1]]
    
    # Inverse frequency weighting
    total = class_0_count + class_1_count
    class_weight = {
        0: total / (2 * class_0_count),
        1: total / (2 * class_1_count)
    }
    print(f"\n⚖️ Class Weights: {class_weight}")
else:
    class_weight = None

# Apply preprocessing and augmentation
def prepare_dataset(ds, augment=False, preprocess=True):
    if preprocess:
        ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), 
                    num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(buffer_size=tf.data.AUTOTUNE)

# Prepare datasets (augment only training data)
train_ds = prepare_dataset(train_ds, augment=True, preprocess=True)
val_ds = prepare_dataset(val_ds, augment=False, preprocess=True)
test_ds = prepare_dataset(test_ds, augment=False, preprocess=True)

print("\n✅ Datasets prepared with augmentation and preprocessing")

In [ ]:
# =================================================================================
# 6. MODEL BUILDING (Enhanced Architecture)
# =================================================================================
print("🏗️ Building Model...")

with strategy.scope():
    # Load pre-trained EfficientNetB0
    base_model = EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        drop_connect_rate=0.2  # Regularization
    )
    
    # Freeze base model initially
    base_model.trainable = False
    
    # Build classification head
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)  # Stabilize training
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(1, activation='sigmoid', dtype='float32')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='deepfake_detector')
    
    # Compile model
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall')
        ]
    )

print("\n📋 Model Summary:")
model.summary()
print(f"\n✅ Total Parameters: {model.count_params():,}")
print(f"   Trainable: {sum([tf.keras.backend.count_params(w) for w in model.trainable_weights]):,}")
print(f"   Non-trainable: {sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights]):,}")

In [ ]:
# =================================================================================
# 7. INITIAL TRAINING (Feature Extraction)
# =================================================================================
print("\n" + "="*60)
print("🚀 PHASE 1: Training Classification Head")
print("="*60)

# Callbacks for optimal training
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        'best_model_phase1.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

# Train with class weights for balanced learning
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks,
    class_weight=class_weight,
    verbose=1
)

print("\n✅ Phase 1 Training Complete!")
print(f"   Best Val Accuracy: {max(history.history['val_accuracy'])*100:.2f}%")
print(f"   Best Val AUC: {max(history.history['val_auc']):.4f}")

In [ ]:
# =================================================================================
# 8. FINE-TUNING (Unlock EfficientNet layers)
# =================================================================================
print("\n" + "="*60)
print("🔥 PHASE 2: Fine-Tuning EfficientNet Layers")
print("="*60)

with strategy.scope():
    # Unfreeze the base model
    base_model.trainable = True
    
    # Freeze bottom layers (keep first 100 layers frozen)
    # Only train top layers which are more task-specific
    for layer in base_model.layers[:100]:
        layer.trainable = False
    
    print(f"   Trainable layers: {sum([layer.trainable for layer in base_model.layers])}")
    
    # Recompile with lower learning rate
    model.compile(
        optimizer=Adam(learning_rate=1e-5),  # Very low LR for fine-tuning
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall')
        ]
    )

# Update callbacks
callbacks_finetune = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-8,
        verbose=1
    ),
    ModelCheckpoint(
        'best_model_phase2.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

# Fine-tune the model
history_fine = model.fit(
    train_ds,
    epochs=FINE_TUNE_EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks_finetune,
    class_weight=class_weight,
    verbose=1
)

print("\n✅ Phase 2 Fine-Tuning Complete!")
print(f"   Best Val Accuracy: {max(history_fine.history['val_accuracy'])*100:.2f}%")
print(f"   Best Val AUC: {max(history_fine.history['val_auc']):.4f}")

In [ ]:
# =================================================================================
# 9. FINAL EVALUATION ON TEST SET
# =================================================================================
print("\n" + "="*60)
print("📊 FINAL EVALUATION")
print("="*60)

test_loss, test_acc, test_auc, test_precision, test_recall = model.evaluate(test_ds, verbose=1)

print(f"\n🎯 Test Results:")
print(f"   Accuracy:  {test_acc*100:.2f}%")
print(f"   AUC:       {test_auc:.4f}")
print(f"   Precision: {test_precision:.4f}")
print(f"   Recall:    {test_recall:.4f}")
print(f"   F1-Score:  {2*(test_precision*test_recall)/(test_precision+test_recall):.4f}")

# Save the final model
model.save('deepfake_detector.keras')
print("\n✅ Model saved as 'deepfake_detector.keras'")

# Also save in H5 format for compatibility
model.save('deepfake_detector.h5')
print("✅ Model saved as 'deepfake_detector.h5'")

# Save label mapping info
with open('label_mapping.txt', 'w') as f:
    f.write(f"Label 0 (prediction < 0.5): {class_names[0]}\n")
    f.write(f"Label 1 (prediction > 0.5): {class_names[1]}\n")
print("✅ Label mapping saved")

In [ ]:
# =================================================================================
# 10. TRAINING VISUALIZATION
# =================================================================================
print("\n📈 Generating Training Plots...")

# Combine both training phases
all_acc = history.history['accuracy'] + history_fine.history['accuracy']
all_val_acc = history.history['val_accuracy'] + history_fine.history['val_accuracy']
all_loss = history.history['loss'] + history_fine.history['loss']
all_val_loss = history.history['val_loss'] + history_fine.history['val_loss']
all_auc = history.history['auc'] + history_fine.history['auc']
all_val_auc = history.history['val_auc'] + history_fine.history['val_auc']

epochs_range = range(len(all_acc))

plt.figure(figsize=(18, 5))

# Accuracy Plot
plt.subplot(1, 3, 1)
plt.plot(epochs_range, all_acc, label='Training Accuracy', linewidth=2)
plt.plot(epochs_range, all_val_acc, label='Validation Accuracy', linewidth=2)
plt.axvline(x=len(history.history['accuracy']), color='r', linestyle='--', label='Fine-tuning Start')
plt.legend(loc='lower right')
plt.title('Model Accuracy', fontsize=14, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.grid(True, alpha=0.3)

# Loss Plot
plt.subplot(1, 3, 2)
plt.plot(epochs_range, all_loss, label='Training Loss', linewidth=2)
plt.plot(epochs_range, all_val_loss, label='Validation Loss', linewidth=2)
plt.axvline(x=len(history.history['loss']), color='r', linestyle='--', label='Fine-tuning Start')
plt.legend(loc='upper right')
plt.title('Model Loss', fontsize=14, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)

# AUC Plot
plt.subplot(1, 3, 3)
plt.plot(epochs_range, all_auc, label='Training AUC', linewidth=2)
plt.plot(epochs_range, all_val_auc, label='Validation AUC', linewidth=2)
plt.axvline(x=len(history.history['auc']), color='r', linestyle='--', label='Fine-tuning Start')
plt.legend(loc='lower right')
plt.title('Model AUC', fontsize=14, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('AUC')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Plots saved as 'training_history.png'")

In [ ]:
# =================================================================================
# 11. TEST PREDICTIONS & CONFUSION MATRIX
# =================================================================================
print("\n🔍 Analyzing Model Predictions...")

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Get predictions on test set
y_true = []
y_pred = []

for images, labels in test_ds:
    predictions = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend((predictions > 0.5).astype(int).flatten())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Classification Report
print("\n📋 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# Sample predictions visualization
print("\n🖼️ Sample Predictions:")
sample_batch = next(iter(test_ds.take(1)))
sample_images, sample_labels = sample_batch
sample_predictions = model.predict(sample_images, verbose=0)

plt.figure(figsize=(15, 10))
for i in range(min(9, len(sample_images))):
    plt.subplot(3, 3, i+1)
    plt.imshow(sample_images[i].numpy().astype("uint8"))
    
    true_label = class_names[int(sample_labels[i])]
    pred_label = class_names[int(sample_predictions[i] > 0.5)]
    confidence = sample_predictions[i][0] if sample_predictions[i] > 0.5 else 1 - sample_predictions[i][0]
    
    color = 'green' if true_label == pred_label else 'red'
    plt.title(f"True: {true_label}\nPred: {pred_label} ({confidence*100:.1f}%)", 
              color=color, fontweight='bold')
    plt.axis('off')

plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Analysis complete!")

## 🎉 Training Complete!

### 📦 Generated Files:
- `deepfake_detector.keras` - Main model file (use this in app.py)
- `deepfake_detector.h5` - Alternative format
- `label_mapping.txt` - Label information
- `training_history.png` - Training curves
- `confusion_matrix.png` - Performance visualization
- `sample_predictions.png` - Example predictions

### 📥 Next Steps:
1. Download `deepfake_detector.keras` from Kaggle/Colab
2. Download `label_mapping.txt` to verify label order
3. Replace the old model file in your app directory
4. Run the Streamlit app: `streamlit run app.py`

### ⚠️ Important Note:
Check `label_mapping.txt` to ensure your app.py uses correct label interpretation!
- If FAKE is label 0: prediction < 0.5 → FAKE
- If REAL is label 0: prediction < 0.5 → REAL